# Incident Response Runbook: DEXX Centralized Key Store Breach

**Tactic:** Credential Access → Exfiltration
**Technique:** T1552.001 (Credentials in Files) + T1041 (Exfiltration Over C2 Channel)
**Severity:** CRITICAL

## Overview

This runbook covers the DEXX platform breach (November 2024), in which an attacker compromised
DEXX servers and discovered that private keys were stored in plaintext in an accessible API
endpoint. The attacker bulk-extracted keys for 8,612 user wallets and drained them simultaneously,
resulting in approximately $21M in losses.

## MITRE ATT&CK Mapping

| Technique | ID | Description |
|---|---|---|
| Credentials in Files | **T1552.001** | Private keys stored in plaintext on server-side storage |
| Exfiltration Over C2 Channel | **T1041** | Bulk key extraction via compromised API endpoint |
| Exploit Public-Facing Application | **T1190** | Server-side application vulnerability used for initial access |
| Data from Information Repositories | **T1213** | Key storage layer treated as an information repository |

## Lateral Movement Analysis

The DEXX breach illustrates how custody architecture becomes a single point of lateral movement:

1. **Server compromise** — Attacker exploits a vulnerability in the DEXX backend (likely exposed API or RCE)
2. **Discovery** — Attacker discovers that private keys are accessible via an internal API endpoint in plaintext
3. **Key storage layer access** — Single API call enumerates all 8,612 user wallet keys
4. **Simultaneous drain** — All wallets drained in parallel within a single block window
5. **No lateral hopping needed** — Centralized key storage means one server compromise = access to all users

**Full lateral movement chain:**
`server vulnerability → backend API access → plaintext key storage layer → 8,612 wallet keypairs → simultaneous drain`

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Identify the server entry point
- Enumerate which wallets were drained and when
- Assess whether key exfiltration is ongoing
- Determine if any keys remain at risk


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

print("=" * 60)
print("STEP 1: Detection & Analysis — DEXX Key Store Breach")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
splunk_indicators = []
unique_users = set()
source_hosts = set()

# Detect bulk API key retrieval
print("\n[QUERY] Searching for bulk key retrieval from internal API...")
splunk_query = '''
index=app_logs OR index=api_access
(uri_path="/api/wallet/key" OR uri_path="/api/keypair*" OR uri_path="/api/private*")
| stats count as request_count, dc(wallet_address) as unique_wallets by src_ip, user_agent, _time
| where request_count > 100 OR unique_wallets > 10
| sort -request_count
'''
try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-48h")
    print(f"   Found {len(splunk_results)} anomalous bulk key retrieval events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

for event in splunk_results:
    system_info = {
        'hostname': event.get('src_ip', 'unknown'),
        'unique_wallets': event.get('unique_wallets', 0),
        'request_count': event.get('request_count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    source_hosts.add(event.get('src_ip', 'unknown'))
    splunk_indicators.append({
        'type': 'bulk_key_retrieval',
        'value': f"src={event.get('src_ip')} wallets={event.get('unique_wallets')} requests={event.get('request_count')}",
        'context': 'Mass private key enumeration via API'
    })

# Detect simultaneous mass withdrawals on-chain
print("\n[QUERY] Detecting simultaneous mass wallet drains on-chain...")
drain_query = '''
index=onchain_events OR index=solana_rpc
instruction="transfer" OR instruction="transferChecked"
| bin _time span=1m
| stats dc(from_wallet) as simultaneous_drains, sum(amount_sol) as total_sol by _time
| where simultaneous_drains > 100
| sort -total_sol
'''
try:
    drain_results = splunk.search_events(drain_query, timeframe="-48h")
    print(f"   Found {len(drain_results)} mass simultaneous drain windows")
    for r in drain_results:
        splunk_indicators.append({
            'type': 'simultaneous_drain',
            'value': f"drains={r.get('simultaneous_drains')} total={r.get('total_sol')} SOL at {r.get('_time')}",
            'context': 'Mass simultaneous wallet drain — characteristic of bulk key exfiltration'
        })
except Exception as e:
    print(f"   Drain detection query failed: {e}")

# Check CrowdStrike for server compromise indicators
print("\n[QUERY] Checking CrowdStrike for server compromise indicators...")
try:
    cs_detections = crowdstrike.get_detections(filter="tactic:'Credential Access'", start_time="-48h")
    for detection in cs_detections:
        affected_systems.append({
            'hostname': detection.get('hostname', ''),
            'device_id': detection.get('device_id', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', '')
        })
    print(f"   CrowdStrike: {len(cs_detections)} credential access detections on servers")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")

# MISP enrichment
print("\n[ENRICHMENT] Checking MISP for attacker infrastructure...")
misp_results = []
try:
    for host in source_hosts:
        hits = misp.search_iocs(host)
        if hits:
            misp_results.extend(hits)
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

print("\n[CASE] Creating IRIS incident case...")
try:
    incident_id = iris.create_case({
        'title': f'DEXX Key Store Breach — {len(unique_users)} wallets — plaintext credential exfiltration',
        'severity': 'CRITICAL',
        'technique': 'T1552.001 + T1041',
        'indicators': splunk_indicators
    })
    print(f"   IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n✅ Detection complete:")
print(f"   - Bulk API events detected: {len([i for i in splunk_indicators if i['type']=='bulk_key_retrieval'])}")
print(f"   - Simultaneous drain windows: {len([i for i in splunk_indicators if i['type']=='simultaneous_drain'])}")
print(f"   - Server compromise detections: {len([s for s in affected_systems if s.get('device_id')])}")
print(f"   - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Take the private key API endpoint offline immediately
- Halt all withdrawal operations platform-wide
- Isolate compromised backend servers
- Alert all 8,612 affected users


In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
disabled_accounts = []
blocked_ips = []

# 1. Kill private key API endpoint
print("\n[CONTAINMENT] Taking private key API endpoint offline...")
try:
    result = shuffle.disable_api_endpoint('/api/wallet/key')
    if result:
        containment_actions.append({'action': 'api_endpoint_disable', 'target': '/api/wallet/key', 'status': 'success', 'timestamp': containment_time})
        print("   ✅ Key API endpoint disabled")
    result2 = shuffle.disable_api_endpoint('/api/keypair')
    if result2:
        containment_actions.append({'action': 'api_endpoint_disable', 'target': '/api/keypair', 'status': 'success', 'timestamp': containment_time})
        print("   ✅ Keypair API endpoint disabled")
except Exception as e:
    print(f"   CRITICAL: Endpoint disable failed — manual firewall rule required: {e}")

# 2. Halt all platform withdrawals
print("\n[CONTAINMENT] Halting all platform withdrawal operations...")
try:
    halt_result = shuffle.halt_withdrawals(platform='dexx')
    if halt_result:
        containment_actions.append({'action': 'withdrawal_halt', 'target': 'dexx platform', 'status': 'success', 'timestamp': containment_time})
        print("   ✅ All withdrawals halted")
except Exception as e:
    print(f"   Withdrawal halt failed: {e}")

# 3. Isolate compromised servers
print("\n[CONTAINMENT] Isolating compromised backend servers...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.isolate_host(system['device_id'])
            if result:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({'action': 'server_isolation', 'target': system['hostname'], 'status': 'success', 'timestamp': containment_time})
                print(f"   Isolated server: {system['hostname']}")
except Exception as e:
    print(f"   Server isolation failed: {e}")

# 4. Block attacker IPs at load balancer
print("\n[CONTAINMENT] Blocking attacker IPs at network perimeter...")
try:
    for host in source_hosts:
        ip_match = re.search(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', host)
        if ip_match:
            result = shuffle.block_ip(ip_match.group())
            if result:
                blocked_ips.append(ip_match.group())
                containment_actions.append({'action': 'ip_block', 'target': ip_match.group(), 'status': 'success', 'timestamp': containment_time})
                print(f"   Blocked attacker IP: {ip_match.group()}")
except Exception as e:
    print(f"   IP blocking failed: {e}")

print(f"\n✅ Containment complete:")
print(f"   - Key API endpoints offline: ✓")
print(f"   - Withdrawals halted: ✓")
print(f"   - Servers isolated: {len(isolated_hosts)}")
print(f"   - Attacker IPs blocked: {len(blocked_ips)}")


## Phase 3: Eradication

### Objectives
- Remove plaintext key storage entirely from server infrastructure
- Patch server vulnerability that enabled initial access
- Migrate to HSM or MPC-based key management
- Wipe all plaintext key material from servers


In [ ]:
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
cleaned_systems = []

# 1. Remove plaintext key files from servers
print("\n[ERADICATION] Removing plaintext private key storage from servers...")
key_removal_script = '''
#!/bin/bash
# CAUTION: Run ONLY after all keys have been migrated to HSM/MPC
# Securely delete all plaintext key files
find /app/data/wallets -name "*.json" -name "*.key" -name "*.pem" | xargs shred -vzu
find /tmp -name "*keypair*" -name "*private*" | xargs shred -vzu
# Wipe database columns containing plaintext keys
psql $DATABASE_URL -c "UPDATE user_wallets SET private_key = NULL WHERE private_key IS NOT NULL;"
# Verify removal
grep -r "privateKey\|secretKey\|ed25519" /app/data/ && echo "WARNING: Keys still found" || echo "OK: No plaintext keys found"
'''
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.run_script(system['device_id'], key_removal_script)
            if result:
                cleaned_systems.append(system['hostname'])
                eradication_actions.append({'action': 'plaintext_key_removal', 'target': system['hostname'], 'status': 'success', 'timestamp': eradication_time})
                print(f"   Plaintext keys removed from: {system['hostname']}")
except Exception as e:
    print(f"   Key removal failed: {e}")

# 2. Patch initial access vulnerability
print("\n[ERADICATION] Patching server vulnerability...")
try:
    vuln_results = crowdstrike.get_vulnerabilities(filter="severity:high,critical", hosts=[s.get('device_id') for s in affected_systems if s.get('device_id')])
    for vuln in vuln_results:
        print(f"   Patching: {vuln.get('cve_id','unknown')} on {vuln.get('hostname','unknown')}")
        crowdstrike.remediate_vulnerability(vuln.get('vuln_id'))
        eradication_actions.append({'action': 'vuln_patch', 'target': vuln.get('cve_id','unknown'), 'status': 'patched', 'timestamp': eradication_time})
    print(f"   Patched {len(vuln_results)} vulnerabilities")
except Exception as e:
    print(f"   Vulnerability patching failed: {e}")

# 3. Verify no residual plaintext keys
print("\n[ERADICATION] Verifying no residual plaintext key storage...")
verify_query = '''
index=app_logs (uri_path="/api/wallet/key" OR uri_path="/api/keypair")
| stats count by src_ip, _time
'''
try:
    verify_results = splunk.search_events(verify_query, timeframe="-1h")
    if not verify_results:
        print("   ✅ No further key API access detected")
    else:
        print(f"   ⚠️  Residual key API access: {len(verify_results)} events")
except Exception as e:
    print(f"   Verification failed: {e}")

print(f"\n✅ Eradication complete:")
print(f"   - Systems with plaintext keys cleaned: {len(cleaned_systems)}")
print(f"   - Vulnerabilities patched ✓")
print(f"   - Key API endpoints verified offline ✓")


## Phase 4: Recovery

### Objectives
- Deploy HSM / MPC-based key management architecture
- Re-enable platform with non-custodial model or hardware-secured custody
- Migrate users to new key infrastructure
- Validate new architecture against same attack vector


In [ ]:
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
restored_services = []

# 1. Deploy MPC key management infrastructure
print("\n[RECOVERY] Deploying MPC/HSM key management infrastructure...")
try:
    mpc_deploy = shuffle.deploy_service('mpc_key_manager', config={
        'provider': 'fireblocks_or_privy',
        'key_shares': 3,
        'threshold': 2,
        'hsm_backed': True,
        'no_single_server_reconstruction': True
    })
    if mpc_deploy:
        restored_services.append('mpc_key_manager')
        recovery_actions.append({'action': 'mpc_deployment', 'target': 'MPC key manager', 'status': 'success', 'timestamp': recovery_time})
        print("   ✅ MPC key management deployed")
    print("   Architecture: keys split across 3 parties, 2-of-3 threshold, no server ever sees full key")
except Exception as e:
    print(f"   MPC deployment failed: {e}")

# 2. Re-enable platform with safe withdrawal flows
print("\n[RECOVERY] Re-enabling platform withdrawals via new secure flow...")
try:
    resume_result = shuffle.resume_withdrawals(platform='dexx', require_mpc_sign=True)
    if resume_result:
        restored_services.append('dexx_withdrawals')
        recovery_actions.append({'action': 'withdrawals_resumed', 'target': 'dexx platform', 'status': 'success', 'timestamp': recovery_time})
        print("   ✅ Withdrawals resumed with MPC signing requirement")
except Exception as e:
    print(f"   Withdrawal resumption failed: {e}")

# 3. Re-enable isolated servers after patching
print("\n[RECOVERY] Re-enabling isolated servers after vulnerability patching...")
try:
    for host in isolated_hosts:
        system = next((s for s in affected_systems if s.get('hostname') == host), None)
        if system and system.get('device_id'):
            result = crowdstrike.reenable_host(system['device_id'])
            if result:
                recovery_actions.append({'action': 'server_reenable', 'target': host, 'status': 'success', 'timestamp': recovery_time})
                print(f"   Re-enabled: {host}")
except Exception as e:
    print(f"   Server re-enable failed: {e}")

# 4. Validate new architecture
print("\n[RECOVERY] Validating new custody architecture...")
try:
    pentest_result = shuffle.run_penetration_test('dexx_key_api', test_cases=[
        'plaintext_key_endpoint_probe',
        'bulk_key_enumeration',
        'server_side_key_reconstruction'
    ])
    for test in pentest_result.get('results', []):
        status = "✅ PASS" if not test.get('vulnerable') else "❌ FAIL"
        print(f"   {status}: {test.get('test_name')}")
    recovery_actions.append({'action': 'architecture_pentest', 'status': 'completed', 'timestamp': recovery_time})
except Exception as e:
    print(f"   Architecture validation failed: {e}")

print(f"\n✅ Recovery complete:")
print(f"   - MPC key management deployed: ✓")
print(f"   - Platform withdrawals restored: ✓")
print(f"   - Architecture validated: ✓")


## Phase 5: Post-Incident Activities

### Objectives
- Publish post-mortem with architecture remediation plan
- Notify all 8,612 affected users and initiate compensation process
- Share attacker infrastructure IOCs with Solana security community


In [ ]:
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

print("\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': 'DEXX Centralized Key Store Breach — IR Report',
        'severity': 'CRITICAL',
        'technique': 'T1552.001 + T1041',
        'root_cause': 'Plaintext private keys stored server-side in accessible API endpoint',
        'affected_wallets': 8612,
        'estimated_loss_usd': 21000000,
        'timeline': {'detection': detection_time, 'containment': containment_time, 'eradication': eradication_time, 'recovery': recovery_time, 'closure': closure_time},
        'recommendations': [
            'Never store private keys server-side in plaintext — use MPC or HSM exclusively',
            'Treat key storage endpoints as highest-security surfaces requiring WAF + anomaly detection',
            'Bulk key retrieval (>1 key per request) should be architecturally impossible',
            'Consider non-custodial model — users hold their own keys',
            'Rate-limit and audit all wallet API endpoints regardless of internal vs external access',
            'Regular penetration testing specifically targeting key storage surfaces'
        ]
    }
    report_filename = f"dexx_platform_hack_report_{incident_id}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)
    print(f"   Report written: {report_filename}")
    post_incident_actions.append({'action': 'report_generation', 'target': report_filename, 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   Report generation failed: {e}")

print("\n[POST-INCIDENT] Sharing IOCs...")
try:
    for host in source_hosts:
        misp.share_indicator({'type': 'ip', 'value': host, 'context': 'DEXX breach attacker infrastructure'}, incident_id)
    print(f"   Shared {len(source_hosts)} IOCs with MISP")
    post_incident_actions.append({'action': 'ioc_sharing', 'target': f'{len(source_hosts)} IPs', 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   IOC sharing failed: {e}")

print("\n[POST-INCIDENT] Closing incident case...")
try:
    iris.close_case(incident_id, {'status': 'closed', 'resolution': 'Migrated to MPC key management, plaintext storage eliminated'})
    print(f"   IRIS case closed: {incident_id}")
except Exception as e:
    print(f"   Case closure failed: {e}")

print(f"\n✅ Post-incident activities complete")
print(f"\n🔒 DEXX Platform Hack IR Complete")


## Summary

The DEXX breach demonstrates that centralized custodial key storage creates a catastrophic
single point of failure: one server compromise equals every user's wallet drained simultaneously.

### Key Takeaways
- Custodial platforms must never store private keys in plaintext — use MPC or HSM exclusively
- Bulk key retrieval should be architecturally impossible (no API endpoint should return multiple private keys)
- Server-side key reconstruction should require threshold signatures across multiple independent parties
- Consider a non-custodial model where possible — users controlling their own keys eliminates this attack class

### Preventive Architecture
- MPC (Multi-Party Computation): keys split across multiple servers/providers, never reconstructed on a single machine
- HSM (Hardware Security Module): keys generated and stored in tamper-resistant hardware
- Non-custodial: platform never holds user keys — users sign locally or via hardware wallet


## References

- https://rekt.news/dexx-rekt/ — Rekt.news DEXX incident analysis
- https://attack.mitre.org/techniques/T1552/001/ — MITRE T1552.001: Credentials in Files
- https://attack.mitre.org/techniques/T1041/ — MITRE T1041: Exfiltration Over C2 Channel
- https://docs.fireblocks.com/platform/fireblocks-mpc/ — Fireblocks MPC-CMP architecture
- https://developers.privy.io/docs/security — Privy embedded wallet security model
- https://csrc.nist.gov/publications/detail/fips/140/3/final — NIST FIPS 140-3 HSM standards
